In [ ]:
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

# 1. 데이터 불러오기
df = pd.read_excel('/Users/qwergs/Desktop/공모전 관련자료/교육부/전처리 파일/마스터데이터.xlsx')

# 2. 결측치 제거
df = df.dropna(subset=['범죄율', '유흥업소밀집지역_대안비율'])

# 3. 종속변수, 독립변수 분리
y = df['범죄율']

# 4. 주요 변수 + 자치구 더미 인코딩 포함
feature_cols = [
    '년도',
    '유흥업소밀집지역_대안비율', '유흥업소밀집지역_비율',
    '학교정화(절대)_대안비율', '학교정화(상대)_대안비율', '아파트지역_대안비율', '주택가주변_대안비율'
]

X_base = df[feature_cols]
X_geo = pd.get_dummies(df['자치구'], prefix='자치구', drop_first=True)
X = pd.concat([X_base, X_geo], axis=1)

# 5. 스케일링
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# 6. 회귀모델 적합
X_const = sm.add_constant(X_scaled)
model = sm.OLS(y.astype(float), X_const).fit()

# 8. 후진 제거법 함수 정의
def backward_elimination(X, y, threshold=0.05):
    X_ = sm.add_constant(X.copy())
    while True:
        model = sm.OLS(y, X_).fit()
        pvalues = model.pvalues.drop('const', errors='ignore')
        max_pval = pvalues.max()
        if max_pval > threshold:
            worst_feature = pvalues.idxmax()
            print(f"제거 중: {worst_feature} (p-value: {max_pval:.4f})")
            X_ = X_.drop(columns=[worst_feature])
        else:
            break
    return model, X_

# 9. 후진 제거법 실행
model_final, X_final = backward_elimination(X_scaled, y)

# 10. 최종 결과 출력
print(model_final.summary())


- R-squared 및 모델 적합도 :  나쁘지 않은 수준 
- F 통계량 : 매우 유의미한 회귀모형 

In [ ]:
df.columns

'년도', '자치구', '연도별_자치구_업소수', '결혼예식장주변', '기타', '아파트지역', '유흥업소밀집지역',
       '주택가주변', '학교정화(상대)', '학교정화(절대)', '결혼예식장주변_비율', '기타_비율', '아파트지역_비율',
       '주택가주변_비율',  '유효_주변구분_업소수',
       '결혼예식장주변_대안비율', '기타_대안비율', 
       '살인', '강도',
       '강간·추행', '절도', '폭력', '소계', '청소년수'

# 다중공선성 점검

In [ ]:
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 1. 사용할 독립변수 선택 (자유롭게 조정 가능)
feature_cols = [
    '주택가주변_비율', '결혼예식장주변_대안비율','기타_대안비율',
    '유흥업소밀집지역','아파트지역_비율','기타_비율','결혼예식장주변_비율',
    '살인', '강도','강간·추행', '절도', '폭력', '소계', '청소년수'
]

# 2. 데이터셋 구성
X = df[feature_cols].dropna()
y = df.loc[X.index, '범죄율']

# 3. 스케일링 (해석엔 큰 영향 없지만 수치 안정성을 위해)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# 4. 상수항 추가 (statsmodels 요구사항)
X_scaled_const = sm.add_constant(X_scaled)

# 5. VIF 계산
vif_df = pd.DataFrame()
vif_df["변수명"] = X_scaled_const.columns
vif_df["VIF"] = [variance_inflation_factor(X_scaled_const.values, i)
                 for i in range(X_scaled_const.shape[1])]

# 6. 결과 출력
print(vif_df.sort_values("VIF", ascending=False))

### **결과**
- 유흥업소밀집지역_대안비율  10.188971 
- 유흥업소밀집지역_비율   5.517093 
- 결혼예식장주변_대안비율  34.988949
- 결혼예식장주변_비율  34.351164
- 기타_비율   7.923885
- 기타_대안비율   6.458191 

In [ ]:
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

# 1. 사용할 변수 정의
feature_cols = ['업태_다양성지수', '청소년수']

# 2. 결측치 제거 (종속변수와 독립변수 모두에서)
df_valid = df.dropna(subset=['범죄율'] + feature_cols)

# 3. 종속변수와 독립변수 분리
y = df_valid['범죄율']
X_base = df_valid[feature_cols]

# 4. 자치구 더미 인코딩 (첫 항목은 기준으로 제거)
X_geo = pd.get_dummies(df_valid['자치구'], prefix='자치구', drop_first=True)

# 5. 전체 독립변수 결합
X = pd.concat([X_base, X_geo], axis=1)

# 6. 스케일링
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# 7. 회귀모형 적합
X_const = sm.add_constant(X_scaled)
model = sm.OLS(y, X_const).fit()

# 8. 결과 출력
print(model.summary())

1.	청소년 인구수는 범죄율과 강한 양의 상관관계를 보입니다.
- 청소년 수가 증가할수록 범죄율이 유의미하게 증가합니다.
- 이 결과는 정책적 접근에서도 유용합니다 (예: 청소년 보호 정책 필요성).
2.	업태_다양성지수는 유의하지 않음
- 상점/업종의 다양성이 범죄율과는 직접적인 상관이 없다는 해석 가능.
3.	자치구 간 유의미한 차이 존재
= 기준 자치구에 비해 일부 자치구는 범죄율이 확실히 더 높거나 낮음.
- 공간적 정책(자치구별 맞춤 대응)에 활용 가능.


In [ ]:
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

# 1. 사용할 새 변수 정의
new_feature_cols = [
    '유흥업소밀집지역_비율',
    '학교정화(절대)_대안비율',
    '아파트지역_대안비율',
    '주택가주변_대안비율',
    '영업중_비율'
]

# 2. 결측치 제거
df_valid2 = df.dropna(subset=['범죄율'] + new_feature_cols)

# 3. 종속변수와 독립변수 분리
y2 = df_valid2['범죄율']
X_base2 = df_valid2[new_feature_cols]

# 4. 자치구 더미 인코딩
X_geo2 = pd.get_dummies(df_valid2['자치구'], prefix='자치구', drop_first=True)

# 5. 전체 독립변수 결합
X2 = pd.concat([X_base2, X_geo2], axis=1)

# 6. 스케일링
scaler = StandardScaler()
X2_scaled = pd.DataFrame(scaler.fit_transform(X2), columns=X2.columns)

# 7. 상수항 추가 및 회귀 적합
X2_const = sm.add_constant(X2_scaled)
model2 = sm.OLS(y2, X2_const).fit()

# 8. 결과 출력
print(model2.summary())

In [ ]:
# 회귀 분석 조합 1
features_1 = ['청소년수', '학교정화(절대)_대안비율', '학교정화(상대)_대안비율']
df_valid1 = df.dropna(subset=['범죄율'] + features_1)

y1 = df_valid1['범죄율']
X1_base = df_valid1[features_1]
X1_geo = pd.get_dummies(df_valid1['자치구'], prefix='자치구', drop_first=True)
X1 = pd.concat([X1_base, X1_geo], axis=1)

scaler = StandardScaler()
X1_scaled = pd.DataFrame(scaler.fit_transform(X1), columns=X1.columns)

X1_const = sm.add_constant(X1_scaled)
model1 = sm.OLS(y1, X1_const).fit()

print("\n[조합 1] 청소년수 + 학교정화 관련 변수")
print(model1.summary())

In [ ]:
# 회귀 분석 조합 2
features_2 = ['유흥업소밀집지역_비율', '업태_다양성지수', '주택가주변_대안비율', '아파트지역_대안비율']
df_valid2 = df.dropna(subset=['범죄율'] + features_2)

y2 = df_valid2['범죄율']
X2_base = df_valid2[features_2]
X2_geo = pd.get_dummies(df_valid2['자치구'], prefix='자치구', drop_first=True)
X2 = pd.concat([X2_base, X2_geo], axis=1)

scaler = StandardScaler()
X2_scaled = pd.DataFrame(scaler.fit_transform(X2), columns=X2.columns)

X2_const = sm.add_constant(X2_scaled)
model2 = sm.OLS(y2, X2_const).fit()

print("\n[조합 2] 유흥업소/업태 + 주거 밀도 관련")
print(model2.summary())

**청소년수 + 업태_다양성지수 + 자치구**
- R² = 0.624 / Adj. R² = 0.590
- 전체 설명력 최고.
- 특히 청소년수가 강한 예측력을 보인 것으로 보이며, 자치구 효과가 설명력 상승에 큰 기여를 했을 가능성.

**청소년수 + 학교정화(절대/상대)_대안비율 + 자치구**
- 설명력 R² = 0.580로 양호함.
- 청소년수 단독 사용보다 약간 설명력이 떨어짐 → 학교정화구역 변수는 영향이 약한 것으로 해석됨.
- 하지만 정책적으로 학교 근처 정화구역이 범죄율과 어떻게 연관되는지 탐색 가능성 있음.

**유흥업소/업태/주거 밀도 관련 + 자치구**
- R² = 0.369, 가장 낮은 설명력.
- 유흥업소 밀집도, 주거 밀도, 업태 다양성은 범죄율을 설명하는 데 비교적 영향력이 낮음.
- 정책적 효과는 있을 수 있으나, 직접적인 회귀 분석 설명력은 떨어짐.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from scipy import stats
from sklearn.preprocessing import PowerTransformer
from scipy.stats import boxcox

# 1. 데이터 불러오기
df = pd.read_excel('/Users/qwergs/Desktop/공모전 관련자료/교육부/전처리 파일/마스터데이터.xlsx')

# 2. 결측치 제거
df = df.dropna(subset=['범죄율', '유흥업소밀집지역_대안비율'])


# 3.0 종속변수 변환
# Box-Cox: 반드시 양수여야 함
# df['범죄율'], fitted_lambda = boxcox(df['범죄율'] + 1e-5)
# # Yeo-Johnson: 0 포함 가능
# pt = PowerTransformer(method='yeo-johnson')
# df['범죄율'] = pt.fit_transform(df[['범죄율']])

# 로그 변환
df['범죄율'] = np.log1p(df['범죄율'])
# 3. 종속변수
y = df['범죄율']


# 4. 독립변수 선택
feature_cols = [
       '기타', '아파트지역',
       '주택가주변', '기타_비율',
       '학교정화(상대)_비율',
       '아파트지역_대안비율',
       '업태_다양성지수', '영업중_비율',
        '절도', '폭력', '청소년수'
]
X_base = df[feature_cols]

# 5. 자치구 더미 인코딩
X_geo = pd.get_dummies(df['자치구'], prefix='자치구', drop_first=True)

# 6. 결합
X = pd.concat([X_base, X_geo], axis=1)

# 7. 스케일링
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# 8. 초기 회귀모델 적합
X_const = sm.add_constant(X_scaled)
model = sm.OLS(y, X_const).fit()

# 9. 잔차 계산 및 정규화
residuals = model.resid
z_scores = stats.zscore(residuals)

# 10. 이상치 제거 기준: |Z-score| > 2.5
mask = np.abs(z_scores) <= 2.5
X_scaled_clean = X_scaled[mask]
y_clean = y[mask]

print(f"제거된 이상치 수: {np.sum(~mask)}")

def stepwise_selection(X, y, threshold_in=0.05, threshold_out=0.05, verbose=True):
    included = []
    while True:
        changed = False
        # Forward step
        excluded = list(set(X.columns) - set(included))
        new_pval = pd.Series(index=excluded, dtype=float)

        for new_col in excluded:
            model = sm.OLS(y, sm.add_constant(X[included + [new_col]])).fit()
            new_pval[new_col] = model.pvalues[new_col]

        if not new_pval.empty:
            best_pval = new_pval.min()
            if best_pval < threshold_in:
                best_feature = new_pval.idxmin()
                included.append(best_feature)
                changed = True
                if verbose:
                    print(f'추가: {best_feature} (p-value: {best_pval:.4f})')

        # Backward step
        if included:
            model = sm.OLS(y, sm.add_constant(X[included])).fit()
            pvalues = model.pvalues.drop('const', errors='ignore')
            worst_pval = pvalues.max()
            if worst_pval > threshold_out:
                worst_feature = pvalues.idxmax()
                included.remove(worst_feature)
                changed = True
                if verbose:
                    print(f'제거: {worst_feature} (p-value: {worst_pval:.4f})')

        if not changed:
            break

    model = sm.OLS(y, sm.add_constant(X[included])).fit()
    return model, included

# 12. 이상치 제거 후 회귀 재학습
model_final, X_final = stepwise_selection(X_scaled_clean, y_clean)

# 13. 결과 출력
print(model_final.summary())



# **최종 검정**

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from scipy import stats
from sklearn.preprocessing import PowerTransformer
from scipy.stats import boxcox

# 1. 데이터 불러오기
df = pd.read_excel('/Users/qwergs/Desktop/공모전 관련자료/교육부/전처리 파일/마스터데이터.xlsx')

# 2. 결측치 제거
df = df.dropna(subset=['범죄율', '유흥업소밀집지역_대안비율'])

# 로그 변환
df['범죄율'] = np.log1p(df['범죄율'])

# 3. 종속변수
y = df['범죄율']

# 4. 독립변수 선택
feature_cols = [
    '기타', '유흥업소밀집지역_대안비율',
    '아파트지역_비율',
    '학교정화(상대)_대안비율','유흥업소밀집지역_비율',
    '업태_다양성지수', '영업중_비율', '절도', '폭력'
]
X_base = df[feature_cols]

# 5. 자치구 더미 인코딩
X_geo = pd.get_dummies(df['자치구'], prefix='자치구', drop_first=True)

# 6. 결합
X = pd.concat([X_base, X_geo], axis=1)

# 7. 스케일링
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# 8. 초기 회귀모델 적합
X_const = sm.add_constant(X_scaled)
model = sm.OLS(y, X_const).fit()

# 9. 잔차 계산 및 정규화
residuals = model.resid
z_scores = stats.zscore(residuals)

# 10. 이상치 제거 기준: |Z-score| > 2.5
mask = np.abs(z_scores) <= 2.5
X_scaled_clean = X_scaled[mask]
y_clean = y[mask]

print(f"제거된 이상치 수: {np.sum(~mask)}")

# 11. 후진 제거법 정의
def backward_elimination(X, y, threshold=0.05):
    X_ = sm.add_constant(X.copy())
    while True:
        model = sm.OLS(y, X_).fit()
        pvalues = model.pvalues.drop('const', errors='ignore')
        max_pval = pvalues.max()
        if max_pval > threshold:
            worst_feature = pvalues.idxmax()
            print(f"제거 중: {worst_feature} (p-value: {max_pval:.4f})")
            X_ = X_.drop(columns=[worst_feature])
        else:
            break
    return model, X_

# 12. 이상치 제거 후 회귀 재학습
model_final, X_final = backward_elimination(X_scaled_clean, y_clean)

# 13. 결과 출력
print(model_final.summary())